In [1]:
from perfect_binarisation import *

In [2]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from contextlib import redirect_stdout

def _pad_to_max_length(curves, pad_value=np.nan):
    max_len = max(len(c) for c in curves)
    out = np.full((len(curves), max_len), pad_value, dtype=float)
    for i, c in enumerate(curves):
        out[i, :len(c)] = c
    return out

def run_5_seeds_and_save_curves(
    train_fn,
    train_kwargs=None,
    base_train_kwargs=None,  # optional compat alias
    seeds=(0, 1, 2, 3, 4),
    out_dir="results",
    label="experiment",
    silence_prints=True,
):
    """
    Runner for any train() that returns:
      policy, queries, steps, average_returns

    Saves in out_dir:
      - {label}.png
      - {label}_seed{seed}_curve.npy for each seed
      - {label}_mean_curve.npy, {label}_std_curve.npy, {label}_x_updates.npy
      - {label}_summary.json
      - optional per-seed logs {label}_seed{seed}_log.txt
    """
    if train_kwargs is None:
        train_kwargs = base_train_kwargs
    if train_kwargs is None:
        raise ValueError("Provide train_kwargs (or base_train_kwargs).")

    os.makedirs(out_dir, exist_ok=True)

    per_seed_curves = []
    final_returns = []
    total_steps = []
    total_queries = []

    for seed in seeds:
        kwargs = dict(train_kwargs)
        kwargs["seed"] = seed

        log_path = os.path.join(out_dir, f"{label}_seed{seed}_log.txt")
        print(f"\n=== Running seed {seed} ===")

        if silence_prints:
            with open(log_path, "w") as f, redirect_stdout(f):
                policy, queries, steps, average_returns = train_fn(**kwargs)
        else:
            policy, queries, steps, average_returns = train_fn(**kwargs)

        curve = np.asarray(average_returns, dtype=float)
        per_seed_curves.append(curve)

        final_returns.append(float(curve[-1]) if len(curve) else np.nan)
        total_steps.append(int(steps))
        total_queries.append(int(queries))

        np.save(os.path.join(out_dir, f"{label}_seed{seed}_curve.npy"), curve)

    curves_mat = _pad_to_max_length(per_seed_curves, pad_value=np.nan)
    mean_curve = np.nanmean(curves_mat, axis=0)
    std_curve = np.nanstd(curves_mat, axis=0)

    x_updates = np.arange(len(mean_curve), dtype=float)

    np.save(os.path.join(out_dir, f"{label}_mean_curve.npy"), mean_curve)
    np.save(os.path.join(out_dir, f"{label}_std_curve.npy"), std_curve)
    np.save(os.path.join(out_dir, f"{label}_x_updates.npy"), x_updates)

    summary = {
        "label": label,
        "seeds": list(seeds),
        "train_kwargs": train_kwargs,
        "final_return_per_seed": final_returns,
        "final_return_mean": float(np.nanmean(final_returns)),
        "final_return_std": float(np.nanstd(final_returns)),
        "total_steps_per_seed": total_steps,
        "total_queries_per_seed": total_queries,
        "curve_len_per_seed": [int(len(c)) for c in per_seed_curves],
    }

    with open(os.path.join(out_dir, f"{label}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # Plot (x-axis is update index)
    plt.figure(figsize=(8, 4.8))
    for c in per_seed_curves:
        plt.plot(np.arange(len(c)), c, alpha=0.25, linewidth=1)

    plt.plot(x_updates, mean_curve, linewidth=2, label="mean (5 seeds)")
    plt.fill_between(
        x_updates,
        mean_curve - std_curve,
        mean_curve + std_curve,
        alpha=0.2,
        label="±1 std",
    )

    plt.xlabel("Training update step (index into average_returns)")
    plt.ylabel("Avg true return")
    plt.title(label)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = os.path.join(out_dir, f"{label}.png")
    plt.savefig(fig_path, dpi=200)
    plt.close()

    print("\nSaved:", fig_path)
    print(
        "Final avg true return over seeds:",
        f"{summary['final_return_mean']:.4f} ± {summary['final_return_std']:.4f}",
    )

    return summary


In [3]:
noise_list = [0.1, 0.5, 0.8]

base_kwargs = dict(
    m=50,
    eta=0.5,
    epsilon=0.1,
    grid_size=8,
    danger=[7,1],
    goal=[4,5],
    wall=[2,5],
    horizon=50,
    coins=None,
)

for sparse in [False, True]:
    for noise in noise_list:
        kwargs = dict(base_kwargs)
        kwargs["noise"] = noise
        kwargs["sparse"] = sparse

        label = f"bernoulli_binary_noise{noise}_sparse{sparse}"
        run_5_seeds_and_save_curves(
            train_fn=train,
            train_kwargs=kwargs,  # or base_train_kwargs=kwargs (compat)
            seeds=(0,2,4),
            out_dir="plot1_model_free_bernoulli_binary",
            label=label,
            silence_prints=0,
        )



=== Running seed 0 ===
0
average reward: 4.82048
average feedback: 0.18
100
average reward: 10.11824
average feedback: 0.36
200
average reward: 16.3728
average feedback: 0.4
300
average reward: 17.94048
average feedback: 0.56
400
average reward: 19.718400000000003
average feedback: 0.48
500
average reward: 19.79168
average feedback: 0.52
600
average reward: 19.7288
average feedback: 0.58
700
average reward: 20.3296
average feedback: 0.66
800
average reward: 20.144
average feedback: 0.54
900
average reward: 20.4456
average feedback: 0.68
1000
average reward: 20.277279999999998
average feedback: 0.58
1100
average reward: 20.485599999999998
average feedback: 0.64
1200
average reward: 20.6256
average feedback: 0.62
1300
average reward: 20.94
average feedback: 0.74
1400
average reward: 20.084
average feedback: 0.56
1500
average reward: 20.8152
average feedback: 0.72
1600
average reward: 20.738400000000002
average feedback: 0.68
1700
average reward: 20.906400000000005
average feedback: 0.46

14200
average reward: 21.331999999999997
average feedback: 0.66
14300
average reward: 20.986400000000003
average feedback: 0.54
14400
average reward: 20.757600000000004
average feedback: 0.6
14500
average reward: 20.869600000000002
average feedback: 0.68
14600
average reward: 20.9864
average feedback: 0.62
14700
average reward: 21.0704
average feedback: 0.56
14800
average reward: 20.732799999999997
average feedback: 0.58
14900
average reward: 20.9944
average feedback: 0.66

=== Running seed 2 ===
0
average reward: 4.78144
average feedback: 0.16
100
average reward: 7.100000000000001
average feedback: 0.24
200
average reward: 15.498080000000002
average feedback: 0.4
300
average reward: 17.764
average feedback: 0.68
400
average reward: 19.3496
average feedback: 0.62
500
average reward: 19.820000000000004
average feedback: 0.6
600
average reward: 19.824800000000003
average feedback: 0.5
700
average reward: 19.59408
average feedback: 0.62
800
average reward: 19.8184
average feedback: 0.54
9

13300
average reward: 20.9256
average feedback: 0.6
13400
average reward: 20.877599999999997
average feedback: 0.54
13500
average reward: 20.7296
average feedback: 0.56
13600
average reward: 21.143199999999997
average feedback: 0.6
13700
average reward: 21.34
average feedback: 0.62
13800
average reward: 21.556000000000004
average feedback: 0.66
13900
average reward: 21.045600000000004
average feedback: 0.7
14000
average reward: 21.260000000000005
average feedback: 0.68
14100
average reward: 20.8136
average feedback: 0.7
14200
average reward: 21.039200000000005
average feedback: 0.6
14300
average reward: 21.252000000000002
average feedback: 0.5
14400
average reward: 21.0744
average feedback: 0.68
14500
average reward: 21.187200000000004
average feedback: 0.64
14600
average reward: 20.893600000000003
average feedback: 0.68
14700
average reward: 21.130400000000005
average feedback: 0.66
14800
average reward: 21.2064
average feedback: 0.7
14900
average reward: 20.927200000000003
average fe

12500
average reward: 20.877599999999997
average feedback: 0.76
12600
average reward: 21.0696
average feedback: 0.54
12700
average reward: 21.163200000000007
average feedback: 0.68
12800
average reward: 21.1432
average feedback: 0.56
12900
average reward: 20.7288
average feedback: 0.58
13000
average reward: 21.532800000000005
average feedback: 0.68
13100
average reward: 21.275200000000005
average feedback: 0.6
13200
average reward: 20.9056
average feedback: 0.6
13300
average reward: 21.062400000000004
average feedback: 0.64
13400
average reward: 21.1184
average feedback: 0.66
13500
average reward: 20.901600000000002
average feedback: 0.58
13600
average reward: 21.275200000000005
average feedback: 0.7
13700
average reward: 21.071199999999997
average feedback: 0.62
13800
average reward: 21.444000000000003
average feedback: 0.54
13900
average reward: 21.002399999999998
average feedback: 0.7
14000
average reward: 21.022399999999998
average feedback: 0.68
14100
average reward: 20.6208
avera

11500
average reward: 20.9544
average feedback: 0.56
11600
average reward: 20.825599999999998
average feedback: 0.64
11700
average reward: 21.0944
average feedback: 0.68
11800
average reward: 20.980800000000002
average feedback: 0.58
11900
average reward: 21.1672
average feedback: 0.64
12000
average reward: 21.1352
average feedback: 0.64
12100
average reward: 20.9784
average feedback: 0.68
12200
average reward: 20.773600000000002
average feedback: 0.56
12300
average reward: 21.151200000000006
average feedback: 0.56
12400
average reward: 21.4568
average feedback: 0.48
12500
average reward: 20.801600000000004
average feedback: 0.56
12600
average reward: 20.942400000000003
average feedback: 0.56
12700
average reward: 21.388
average feedback: 0.48
12800
average reward: 20.9344
average feedback: 0.6
12900
average reward: 21.0184
average feedback: 0.54
13000
average reward: 21.0184
average feedback: 0.52
13100
average reward: 21.038400000000003
average feedback: 0.56
13200
average reward: 20

...

9300
average reward: 20.8904
average feedback: 0.54
9400
average reward: 20.052
average feedback: 0.56
9500
average reward: 21.095200000000006
average feedback: 0.46
9600
average reward: 20.6248
average feedback: 0.52
9700
average reward: 20.09568
average feedback: 0.6
9800
average reward: 21.0912
average feedback: 0.44
9900
average reward: 20.8384
average feedback: 0.4
10000
average reward: 20.472800000000003
average feedback: 0.54
10100
average reward: 20.802400000000002
average feedback: 0.56
10200
average reward: 21.132
average feedback: 0.64
10300
average reward: 20.8136
average feedback: 0.44
10400
average reward: 20.882400000000004
average feedback: 0.6
10500
average reward: 21.123200000000008
average feedback: 0.5
10600
average reward: 20.822400000000002
average feedback: 0.54
10700
average reward: 20.8384
average feedback: 0.58
10800
average reward: 20.6776
average feedback: 0.54
10900
average reward: 20.790399999999998
average feedback: 0.52
11000
average reward: 20.738400000

8800
average reward: 20.0024
average feedback: 0.5
8900
average reward: 20.437600000000003
average feedback: 0.58
9000
average reward: 20.4712
average feedback: 0.56
9100
average reward: 20.3472
average feedback: 0.58
9200
average reward: 20.688000000000002
average feedback: 0.66
9300
average reward: 20.5392
average feedback: 0.54
9400
average reward: 20.479200000000002
average feedback: 0.48
9500
average reward: 20.398400000000002
average feedback: 0.48
9600
average reward: 20.554879999999997
average feedback: 0.48
9700
average reward: 20.4584
average feedback: 0.52
9800
average reward: 20.27888
average feedback: 0.6
9900
average reward: 20.387200000000004
average feedback: 0.52
10000
average reward: 20.3656
average feedback: 0.52
10100
average reward: 20.514400000000002
average feedback: 0.56
10200
average reward: 20.651200000000003
average feedback: 0.44
10300
average reward: 20.4664
average feedback: 0.56
10400
average reward: 20.9808
average feedback: 0.52
10500
average reward: 20

8300
average reward: 20.1688
average feedback: 0.58
8400
average reward: 20.304800000000004
average feedback: 0.54
8500
average reward: 20.6584
average feedback: 0.4
8600
average reward: 20.052000000000003
average feedback: 0.48
8700
average reward: 20.008799999999997
average feedback: 0.52
8800
average reward: 20.26
average feedback: 0.5
8900
average reward: 20.328800000000005
average feedback: 0.44
9000
average reward: 20.5952
average feedback: 0.76
9100
average reward: 20.671200000000002
average feedback: 0.58
9200
average reward: 20.108800000000002
average feedback: 0.42
9300
average reward: 20.563200000000002
average feedback: 0.38
9400
average reward: 20.4264
average feedback: 0.54
9500
average reward: 20.514400000000002
average feedback: 0.46
9600
average reward: 20.6152
average feedback: 0.56
9700
average reward: 20.5984
average feedback: 0.44
9800
average reward: 20.597600000000003
average feedback: 0.54
9900
average reward: 20.671200000000002
average feedback: 0.58
10000
aver

3100
average reward: 10.0992
average feedback: 0.72
3200
average reward: 10.639200000000002
average feedback: 0.72
3300
average reward: 11.280000000000003
average feedback: 0.66
3400
average reward: 11.917600000000002
average feedback: 0.74
3500
average reward: 12.375200000000005
average feedback: 0.7

=== Running seed 2 ===
0
average reward: 1.5967199999999997
average feedback: 0.32
100
average reward: 1.3073599999999999
average feedback: 0.26
200
average reward: 2.03856
average feedback: 0.3
300
average reward: 2.60048
average feedback: 0.4
400
average reward: 5.04288
average feedback: 0.52
500
average reward: 5.5232
average feedback: 0.4
600
average reward: 5.975999999999999
average feedback: 0.52
700
average reward: 6.5696
average feedback: 0.48
800
average reward: 7.1432
average feedback: 0.58
900
average reward: 7.7352
average feedback: 0.48
1000
average reward: 10.998400000000002
average feedback: 0.7
1100
average reward: 11.132
average feedback: 0.7
1200
average reward: 11.4912

1100
average reward: 4.6568
average feedback: 0.56
1200
average reward: 6.38104
average feedback: 0.5
1300
average reward: 5.681839999999999
average feedback: 0.54
1400
average reward: 5.6656
average feedback: 0.46
1500
average reward: 5.672000000000001
average feedback: 0.48
1600
average reward: 5.7584
average feedback: 0.46
1700
average reward: 5.59168
average feedback: 0.56
1800
average reward: 5.5256
average feedback: 0.46
1900
average reward: 5.9712
average feedback: 0.52
2000
average reward: 6.004
average feedback: 0.48
2100
average reward: 6.528
average feedback: 0.5
2200
average reward: 6.175999999999999
average feedback: 0.38
2300
average reward: 6.056
average feedback: 0.5
2400
average reward: 6.171199999999999
average feedback: 0.44
2500
average reward: 6.058399999999999
average feedback: 0.54
2600
average reward: 6.530399999999999
average feedback: 0.54
2700
average reward: 6.7712
average feedback: 0.46
2800
average reward: 6.4848
average feedback: 0.54
2900
average reward: